In [1]:
import os
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential
import torch.optim as optim
import voxelmorph as vxm
import neurite as ne
import scipy.ndimage

os.environ['VXM_BACKEND'] = 'pytorch'

backend:pytorch
Pytorch


In [2]:
os.environ.get('VXM_BACKEND')

'pytorch'

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [4]:
# 画像を読み込み
x_train = np.load('Data/TrainData_NoBed.npz')['Train']
x_train = np.transpose(x_train, (3, 0, 1, 2))

print('Resized train vol_shape:', x_train.shape[1:])
print('Resized train shape:', x_train.shape)

Resized train vol_shape: (128, 256, 256)
Resized train shape: (400, 128, 256, 256)


In [5]:
import torch

def vxm_data_generator(x_data, batch_size):
    vol_shape = x_data.shape[1:]  # データ形状を取得
    ndims = len(vol_shape)
    
    zero_phi = np.zeros([batch_size, *vol_shape, ndims])
    
    while True:
        idx1 = np.random.randint(0, x_data.shape[0], size=batch_size)
        moving_images = x_data[idx1, ..., np.newaxis]
        # ファインチューニングでは同じ症例同士のペアを避ける
        idx2 = np.random.randint(0, x_data.shape[0], size=batch_size)
        while np.any(idx2 == idx1):
            same_case = idx2 == idx1
            idx2[same_case] = np.random.randint(0, x_data.shape[0], size=same_case.sum())
        fixed_images = x_data[idx2, ..., np.newaxis]

        # TensorFlowからPyTorchのデータ形式に変換
        moving_images = torch.tensor(moving_images).permute(0, 4, 1, 2, 3).float()
        fixed_images = torch.tensor(fixed_images).permute(0, 4, 1, 2, 3).float()

        # チャンネルを最初の次元に追加
        moving_images = moving_images.permute(0, 1, 2, 3, 4)  # チャンネルを最初の次元に移動
        fixed_images = fixed_images.permute(0, 1, 2, 3, 4)  # チャンネルを最初の次元に移動

        inputs = [moving_images, fixed_images]
        outputs = [fixed_images, zero_phi]

        yield (inputs, outputs)

In [6]:
train_generator = vxm_data_generator(x_train, batch_size=2)
in_sample, out_sample = next(train_generator)

# in_sampleとout_sampleの内容を確認する
print("Input Sample Shapes:")
print("Moving Images Shape:", in_sample[0].shape)
print("Fixed Images Shape:", in_sample[1].shape)

print("\nOutput Sample Shapes:")
print("Moved Images (Fixed) Shape:", out_sample[0].shape)
print("Zero Gradient Shape:", out_sample[1].shape)

Input Sample Shapes:
Moving Images Shape: torch.Size([2, 1, 128, 256, 256])
Fixed Images Shape: torch.Size([2, 1, 128, 256, 256])

Output Sample Shapes:
Moved Images (Fixed) Shape: torch.Size([2, 1, 128, 256, 256])
Zero Gradient Shape: (2, 128, 256, 256, 3)


In [7]:
mse_loss = vxm.losses.MSE().loss
grad_loss = vxm.losses.Grad('l2').loss

def total_loss(y_true, y_pred):
    mse = mse_loss(y_true, y_pred)
    grad = grad_loss(y_true, y_pred)
    return mse + 0.01 * grad, mse, grad
#     return mse_loss(y_true, y_pred)

def MSE_Loss(y_true, y_pred):
    y_true = y_true.to(device)
    y_pred = y_pred.to(device)
    mse = mse_loss(y_true, y_pred)
    return mse

def lncc_loss(I, J, window=9, eps=1e-5):
    # I, J: (B, 1, D, H, W)
    padding = window // 2
    weight = torch.ones(1, 1, window, window, window, device=I.device)

    I2 = I * I
    J2 = J * J
    IJ = I * J

    I_sum = F.conv3d(I, weight, padding=padding)
    J_sum = F.conv3d(J, weight, padding=padding)
    I2_sum = F.conv3d(I2, weight, padding=padding)
    J2_sum = F.conv3d(J2, weight, padding=padding)
    IJ_sum = F.conv3d(IJ, weight, padding=padding)

    win_size = window ** 3
    u_I = I_sum / win_size
    u_J = J_sum / win_size

    cross = IJ_sum - u_J * I_sum - u_I * J_sum + u_I * u_J * win_size
    I_var = I2_sum - 2 * u_I * I_sum + u_I * u_I * win_size
    J_var = J2_sum - 2 * u_J * J_sum + u_J * u_J * win_size

    lncc = cross * cross / (I_var * J_var + eps)
    return -torch.mean(lncc)  # maximize LNCC → minimize -LNCC

In [8]:
# configure unet input shape (concatenation of moving and fixed images)
ndim = 3
unet_input_features = 2
# inshape = (*x_train.shape[1:], unet_input_features)

nb_features = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16]
]


In [9]:
import voxelmorph as vxm
import inspect

print(vxm.__file__)
print(vxm.networks.__file__)
print([name for name in dir(vxm.networks) if "VxmDense" in name])

c:\Users\ri0151fv\Saito\voxelmorph\__init__.py
c:\Users\ri0151fv\Saito\voxelmorph\torch\networks.py
['VxmDense', 'VxmDense1', 'VxmDense2', 'VxmDense_128_256', 'VxmDense_128_256_256']


In [10]:
model3D = vxm.networks.VxmDense_128_256_256((128, 256, 256), nb_features, int_steps=0)
model3D.to(device)
optimizer = optim.Adam(model3D.parameters(), lr=1e-4)

transformer = vxm.layers.SpatialTransformer((64, 128, 128)).to(device)
transformer256 = vxm.layers.SpatialTransformer((128, 256, 256)).to(device)

[64, 128, 128]


c:\Users\ri0151fv\AppData\Local\anaconda3\envs\vxm310\lib\site-packages\torch\functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3596.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [11]:
import math
from pathlib import Path
import torch
import matplotlib.pyplot as plt

band_names = ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']

wavelet_vis_enabled = False
wavelet_vis_every = 100
wavelet_vis_dir = Path('wavelet_stage_outputs')
wavelet_vis_dir.mkdir(exist_ok=True)

class Haar3DAnalysisOnly(nn.Module):
    def __init__(self):
        super().__init__()

        hL = torch.tensor([1.0, 1.0], dtype=torch.float32) / math.sqrt(2.0)
        hH = torch.tensor([1.0, -1.0], dtype=torch.float32) / math.sqrt(2.0)

        filters = []
        names = []

        for z_name, z_filter in zip(['L', 'H'], [hL, hH]):
            for y_name, y_filter in zip(['L', 'H'], [hL, hH]):
                for x_name, x_filter in zip(['L', 'H'], [hL, hH]):
                    kernel = (
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )
                    filters.append(kernel)
                    names.append(z_name + y_name + x_name)

        weight = torch.stack(filters, dim=0).unsqueeze(1)
        self.register_buffer('weight', weight)
        self.names = names

    def forward(self, x):
        x = F.pad(x, (0, 1, 0, 1, 0, 1))
        return F.conv3d(x, self.weight, stride=1, padding=0)

def analysis_filter_3d(x, analysis_layer):
    return analysis_layer(x)

def down_sampling_3d(w):
    return w[:, :, ::2, ::2, ::2]

def up_sampling_3d(w_down):
    B, C, D, H, W = w_down.shape
    w_up = torch.zeros(
        B, C, D * 2, H * 2, W * 2,
        dtype=w_down.dtype,
        device=w_down.device
    )
    w_up[:, :, ::2, ::2, ::2] = w_down
    return w_up

def make_3d_filter(fz, fy, fx):
    return fz[:, None, None] * fy[None, :, None] * fx[None, None, :]

def create_synthesis_filters(device):
    low = torch.tensor([1.0, 1.0], dtype=torch.float32, device=device) / math.sqrt(2.0)
    high = torch.tensor([1.0, -1.0], dtype=torch.float32, device=device) / math.sqrt(2.0)

    filters = torch.stack([
        make_3d_filter(low, low, low),
        make_3d_filter(low, low, high),
        make_3d_filter(low, high, low),
        make_3d_filter(low, high, high),
        make_3d_filter(high, low, low),
        make_3d_filter(high, low, high),
        make_3d_filter(high, high, low),
        make_3d_filter(high, high, high),
    ], dim=0)

    filters = torch.flip(filters, dims=[1, 2, 3]).unsqueeze(1)
    return filters

def synthesis_filter_3d(w_up, synthesis_filters):
    B, C, D, H, W = w_up.shape
    filtered_bands = []

    for i in range(C):
        band = w_up[:, i:i + 1, :, :, :]
        kernel = synthesis_filters[i:i + 1]
        filtered = F.conv3d(band, kernel, stride=1, padding=1)
        filtered = filtered[:, :, :D, :H, :W]
        filtered_bands.append(filtered)

    filtered_bands = torch.cat(filtered_bands, dim=1)
    reconstructed = torch.sum(filtered_bands, dim=1, keepdim=True)
    return reconstructed, filtered_bands

analysis = Haar3DAnalysisOnly().to(device)
synthesis_filters = create_synthesis_filters(device)
analysis_names = analysis.names

In [12]:
# Safety switch: this notebook fine-tunes an existing 80k checkpoint.
# Leave False and skip this cell; Cell 13 starts the corrected fine-tuning.
RUN_COPY1_PRETRAINING = False

if RUN_COPY1_PRETRAINING:
    # NotdecoderHight
    from tqdm.notebook import tqdm
    from IPython.display import clear_output
    import matplotlib.pyplot as plt
    
    def gaussian_smooth_3d(tensor, kernel_size=5, sigma=1.0):
        """ 3D ガウシアンフィルタで displacement field をスムージング """
        from scipy.ndimage import gaussian_filter
        tensor_np = tensor.cpu().numpy()
        smoothed_np = gaussian_filter(tensor_np, sigma=[0, 0, sigma, sigma, sigma])
        return torch.tensor(smoothed_np, dtype=torch.float32, device=tensor.device)
    
    epochs = 80000
    best_loss = float('inf')
    shift_range = 1
    
    losses = []
    loss_vecs = []
    loss_images = []
    loss_hightVecs = []
    for epoch in tqdm(range(epochs)):
        if epoch % 2000 == 0 and epoch > 0:
            shift_range += 1
            print(f"Epoch {epoch}: Increasing shift range to ±{shift_range} pixels.")
    
        train_batch, _ = next(train_generator)
        moving_images = torch.tensor(train_batch[0], dtype=torch.float32).to(device)
    
        B, D, H, W = 2, 8, 16, 16
    
        displacement_field = (torch.rand((B, 3, D, H, W), dtype=torch.float32) * 2 - 1) * shift_range
        displacement_field = displacement_field.to(device)
    
        displacement_field = gaussian_smooth_3d(displacement_field, sigma=2.0)
        displacement_field = torch.nn.functional.interpolate(
            displacement_field,
            size=(128, 256, 256),
            mode='trilinear',
            align_corners=False
        )
        displacement_field128 = torch.nn.functional.interpolate(
            displacement_field,
            size=(64, 128, 128),
            mode='trilinear',
            align_corners=False
        )
    
        moving_images2 = transformer256(moving_images, displacement_field)
    
        # Analysis_Filter.py
        moving_analysis = analysis_filter_3d(moving_images, analysis)
        moving_images2_analysis = analysis_filter_3d(moving_images2, analysis)
    
        # Down_Sampling.py
        moving_w = down_sampling_3d(moving_analysis).to(device)
        moving_images2_w = down_sampling_3d(moving_images2_analysis).to(device)
    
        optimizer.zero_grad()
        Vec = model3D(moving_w, moving_images2_w)
    
        warped_bands = [transformer(moving_w[:, i:i + 1], Vec) for i in range(moving_w.shape[1])]
        moving_warped = torch.cat(warped_bands, dim=1)
    
        # Up_Sampling.py
        moving_warped_up = up_sampling_3d(moving_warped)
    
        # Synthesis_Filter.py
        transformed_image, filtered_bands = synthesis_filter_3d(moving_warped_up, synthesis_filters)
        transformed_image = transformed_image.to(device)
    
        loss_vec = MSE_Loss(displacement_field128, Vec) * 0.01
        loss_image = MSE_Loss(moving_images2, transformed_image) * 100
        loss = loss_vec + loss_image
    
        loss.backward()
        optimizer.step()
    
        if (epoch + 1) % 100 == 0:
            torch.save(model3D.state_dict(), 'model_analysis_pipeline_pretrain.pth')
    
        losses.append(loss.cpu().item())
        loss_vecs.append(loss_vec.cpu().item())
        loss_images.append(loss_image.cpu().item())
    
        if epoch % 10 == 0:
            clear_output(wait=True)
            plt.figure(figsize=(10, 5))
            plt.plot(losses, label='Loss')
            plt.plot(loss_vecs, label='loss_vecs')
            plt.plot(loss_images, label='loss_images')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.title('Training Loss Progress')
            plt.legend()
            plt.grid(True)
            plt.show()
    
        print(
            f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}, loss_vec: {loss_vec:.4f}, "
            f"loss_image: {loss_image:.4f}, Shift Range: ±{shift_range} pixels"
        )
else:
    print('Skipped 80k pretraining. Run Cell 13 for inverse-consistent fine-tuning.')


Skipped 80k pretraining. Run Cell 13 for inverse-consistent fine-tuning.


In [ ]:
# Different-patient lung-aware fine-tuning (Copy1 improved version)
#
# Main changes from Copy1:
#   1. Prefer lung-masked training data and compute image loss only on the overlap.
#   2. Penalize spatially rough displacement fields and clip gradients.
#   3. Use a held-out patient split and select the best checkpoint by validation MSE.
#   4. Save periodic checkpoints rather than overwriting one file every 100 epochs.
#   5. Render moving/fixed/warped slices using the same intensity window.
#   6. Predict both directions and penalize the composed A→B→A / B→A→B flow.

from pathlib import Path
from tqdm import tqdm
from IPython.display import clear_output
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.optim as optim

DATA_DIR = Path('Data')
RAW_DATA_PATH = DATA_DIR / 'TrainData_NoBed.npz'
PRETRAINED_MODEL_PATH = Path('model_analysis_pipeline_pretrain.pth')
CHECKPOINT_DIR = Path('finetune_checkpoints_copy1_lung_inverse_consistent')

# Set this to True when lung fine-tuning must never silently fall back to
# unmasked data.  With False, an already masked archive is also accepted.
REQUIRE_EXPLICIT_LUNG_MASK = False
MASKED_ARCHIVE_CANDIDATES = [
    DATA_DIR / 'TrainData_NoBed_LungMasked.npz',
    DATA_DIR / 'TrainData_NoBed_lung_masked.npz',
    DATA_DIR / 'TrainData_NoBed_Masked.npz',
    DATA_DIR / 'TrainData_NoBed_masked.npz',
]
VOLUME_KEYS = ('Train_lung_masked', 'train_lung_masked', 'TrainMasked', 'masked_train', 'Train')
MASK_KEYS = ('Train_lung_mask', 'train_lung_mask', 'LungMask', 'lung_mask',
             'TrainMask', 'train_mask', 'Mask', 'mask')

FINETUNE_EPOCHS = 30_000
BATCH_SIZE = 2
LEARNING_RATE = 1e-6
SMOOTHNESS_WEIGHT = 0.1
# Flow is in low-resolution voxel units.  Start small so inverse consistency
# improves geometric coherence without overwhelming the image-matching loss.
INVERSE_CONSISTENCY_WEIGHT = 0.01
VALIDATION_INTERVAL = 1_000
VISUALIZATION_INTERVAL = 1_000
VALIDATION_FRACTION = 0.10
SPLIT_SEED = 20260728
PAIR_SEED = 20260729


def to_n_dhw(array, label):
    """Convert a supported layout to (N, 128, 256, 256)."""
    array = np.asarray(array, dtype=np.float32)
    if array.ndim != 4:
        raise ValueError(f'{label}: expected 4-D data, got {array.shape}')
    if array.shape[1:] == (128, 256, 256):
        return array
    if array.shape[:3] == (128, 256, 256):
        return np.transpose(array, (3, 0, 1, 2))
    raise ValueError(f'{label}: unsupported shape {array.shape}; expected N,D,H,W or D,H,W,N')


def first_available_key(archive, candidates):
    return next((key for key in candidates if key in archive.files), None)


def load_lung_preferred_training_data():
    """Load a lung-masked archive if available, otherwise use a mask key in raw data."""
    masked_path = next((path for path in MASKED_ARCHIVE_CANDIDATES if path.exists()), None)
    archive_path = masked_path or RAW_DATA_PATH
    if not archive_path.exists():
        raise FileNotFoundError(f'Training archive was not found: {archive_path.resolve()}')

    with np.load(archive_path, allow_pickle=False) as archive:
        volume_key = first_available_key(archive, VOLUME_KEYS)
        if volume_key is None:
            raise KeyError(f'{archive_path} has no volume key. Available keys: {archive.files}')
        volumes = to_n_dhw(archive[volume_key], f'{archive_path}:{volume_key}')
        mask_key = first_available_key(archive, MASK_KEYS)
        masks = None if mask_key is None else to_n_dhw(archive[mask_key], f'{archive_path}:{mask_key}')

    if masks is not None:
        if masks.shape != volumes.shape:
            raise ValueError(f'Volume/mask shape mismatch: {volumes.shape} vs {masks.shape}')
        masks = (masks > 0).astype(np.float32)
        volumes = volumes * masks
        print(f'Using lung mask key: {mask_key} from {archive_path}')
    elif masked_path is not None:
        print(f'Using pre-masked volume archive: {archive_path}')
    elif REQUIRE_EXPLICIT_LUNG_MASK:
        raise FileNotFoundError(
            'No lung mask was found. Add a lung-mask key to TrainData_NoBed.npz or '
            'place TrainData_NoBed_LungMasked.npz in Data/.'
        )
    else:
        print('WARNING: no lung mask was found; image loss will use all voxels.')
        print('         Put a lung-masked archive in Data/ for true lung-focused fine-tuning.')

    print('Fine-tuning volume shape:', volumes.shape)
    return volumes, masks, archive_path


def make_train_validation_split(n_patients, validation_fraction=0.10, seed=0):
    if n_patients < 4:
        raise ValueError('At least four patients are required for a train/validation split.')
    validation_count = min(max(2, int(round(n_patients * validation_fraction))), n_patients - 2)
    ids = np.random.default_rng(seed).permutation(n_patients)
    validation_ids = np.sort(ids[:validation_count])
    train_ids = np.sort(ids[validation_count:])
    return train_ids, validation_ids


def different_patient_generator(volumes, masks, patient_ids, batch_size=2, seed=0):
    """Yield batches where moving/fixed are always from different patients."""
    patient_ids = np.asarray(patient_ids, dtype=np.int64)
    if len(patient_ids) < 2:
        raise ValueError('At least two training patients are required.')
    rng = np.random.default_rng(seed)
    while True:
        moving_ids = rng.choice(patient_ids, size=batch_size, replace=True)
        fixed_ids = rng.choice(patient_ids, size=batch_size, replace=True)
        while np.any(moving_ids == fixed_ids):
            same = moving_ids == fixed_ids
            fixed_ids[same] = rng.choice(patient_ids, size=int(same.sum()), replace=True)

        moving = torch.from_numpy(volumes[moving_ids]).unsqueeze(1)
        fixed = torch.from_numpy(volumes[fixed_ids]).unsqueeze(1)
        if masks is None:
            overlap = None
        else:
            moving_mask = torch.from_numpy(masks[moving_ids]).unsqueeze(1)
            fixed_mask = torch.from_numpy(masks[fixed_ids]).unsqueeze(1)
            overlap = moving_mask * fixed_mask
        yield moving, fixed, overlap, moving_ids, fixed_ids


def make_fixed_pairs(patient_ids, count=10, seed=0):
    patient_ids = np.asarray(patient_ids, dtype=np.int64)
    if len(patient_ids) < 2:
        raise ValueError('At least two validation patients are required.')
    rng = np.random.default_rng(seed)
    pairs = []
    while len(pairs) < count:
        moving_id, fixed_id = rng.choice(patient_ids, size=2, replace=False)
        pairs.append((int(moving_id), int(fixed_id)))
    return pairs


def masked_mse(target, prediction, mask=None, eps=1e-6):
    squared_error = (target - prediction).square()
    if mask is None:
        return squared_error.mean()
    return (squared_error * mask).sum() / mask.sum().clamp_min(eps)


def smoothness_loss(flow):
    """Mean squared first derivative of the low-resolution displacement field."""
    dz = (flow[:, :, 1:, :, :] - flow[:, :, :-1, :, :]).square().mean()
    dy = (flow[:, :, :, 1:, :] - flow[:, :, :, :-1, :]).square().mean()
    dx = (flow[:, :, :, :, 1:] - flow[:, :, :, :, :-1]).square().mean()
    return (dx + dy + dz) / 3.0


def inverse_consistency_loss(flow_moving_to_fixed, flow_fixed_to_moving):
    """Penalize both round trips after correct displacement-field composition.

    SpatialTransformer samples ``source(p + flow(p))`` and flows are expressed
    in low-resolution voxel coordinates.  Therefore an inverse pair must obey
    ``f_mf + warp(f_fm, f_mf) = 0`` (and its reverse), rather than merely
    ``f_fm = -f_mf`` at the same grid position.
    """
    round_trip_mf = flow_moving_to_fixed + transformer(
        flow_fixed_to_moving, flow_moving_to_fixed
    )
    round_trip_fm = flow_fixed_to_moving + transformer(
        flow_moving_to_fixed, flow_fixed_to_moving
    )
    return 0.5 * (round_trip_mf.square().mean() + round_trip_fm.square().mean())


def reconstruct_warped(moving_images, fixed_images):
    """Run the same Haar/downsample/warp/upsample/synthesis pipeline as Copy1."""
    moving_analysis = analysis_filter_3d(moving_images, analysis)
    fixed_analysis = analysis_filter_3d(fixed_images, analysis)
    moving_w = down_sampling_3d(moving_analysis).to(device)
    fixed_w = down_sampling_3d(fixed_analysis).to(device)
    flow = model3D(moving_w, fixed_w)
    warped_bands = [transformer(moving_w[:, i:i + 1], flow) for i in range(moving_w.shape[1])]
    warped_wavelets = torch.cat(warped_bands, dim=1)
    warped_up = up_sampling_3d(warped_wavelets)
    warped_image, _ = synthesis_filter_3d(warped_up, synthesis_filters)
    return warped_image.to(device), flow


def evaluate_fixed_pairs(pairs):
    model3D.eval()
    mse_values, mae_values, inverse_values = [], [], []
    with torch.no_grad():
        for moving_id, fixed_id in pairs:
            moving = torch.from_numpy(volumes[moving_id:moving_id + 1]).unsqueeze(1).to(device)
            fixed = torch.from_numpy(volumes[fixed_id:fixed_id + 1]).unsqueeze(1).to(device)
            overlap = None if lung_masks is None else torch.from_numpy(
                lung_masks[moving_id:moving_id + 1] * lung_masks[fixed_id:fixed_id + 1]
            ).unsqueeze(1).to(device)
            warped, flow_mf = reconstruct_warped(moving, fixed)
            _, flow_fm = reconstruct_warped(fixed, moving)
            mse_values.append(masked_mse(fixed, warped, overlap).item())
            inverse_values.append(inverse_consistency_loss(flow_mf, flow_fm).item())
            error = (fixed - warped).abs()
            mae_values.append(
                error.mean().item() if overlap is None
                else (error * overlap).sum().div(overlap.sum().clamp_min(1e-6)).item()
            )
    model3D.train()
    return (
        float(np.mean(mse_values)),
        float(np.mean(mae_values)),
        float(np.mean(inverse_values)),
    )


def show_registration(moving, fixed, warped, epoch):
    """Show a truthful side-by-side comparison: one shared CT intensity window."""
    moving_np, fixed_np, warped_np = [x[0, 0].detach().cpu().numpy() for x in (moving, fixed, warped)]
    slice_index = moving_np.shape[0] // 2
    image_vmin, image_vmax = np.percentile(
        np.concatenate([moving_np.ravel(), fixed_np.ravel(), warped_np.ravel()]), [1, 99]
    )
    error_before = np.abs(fixed_np - moving_np)
    error_after = np.abs(fixed_np - warped_np)
    error_vmax = max(np.percentile(np.concatenate([error_before.ravel(), error_after.ravel()]), 99), 1e-6)

    fig, axes = plt.subplots(1, 5, figsize=(18, 4), constrained_layout=True)
    panels = [
        ('Moving', moving_np[slice_index], 'gray', image_vmin, image_vmax),
        ('Fixed', fixed_np[slice_index], 'gray', image_vmin, image_vmax),
        ('Warped', warped_np[slice_index], 'gray', image_vmin, image_vmax),
        ('|Fixed − moving|', error_before[slice_index], 'magma', 0, error_vmax),
        ('|Fixed − warped|', error_after[slice_index], 'magma', 0, error_vmax),
    ]
    for ax, (title, image, cmap, vmin, vmax) in zip(axes, panels):
        im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.axis('off')
        if title.startswith('|'):
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(f'Fine-tuning epoch {epoch}: shared display window', fontsize=14)
    plt.show()


volumes, lung_masks, archive_path = load_lung_preferred_training_data()
train_ids, validation_ids = make_train_validation_split(
    len(volumes), validation_fraction=VALIDATION_FRACTION, seed=SPLIT_SEED
)
validation_pairs = make_fixed_pairs(validation_ids, count=10, seed=PAIR_SEED)
print(f'Train patients: {len(train_ids)}, validation patients: {len(validation_ids)}')
print('Fixed validation pairs:', validation_pairs)

if not PRETRAINED_MODEL_PATH.exists():
    raise FileNotFoundError(f'Pretrained model not found: {PRETRAINED_MODEL_PATH.resolve()}')

model3D = vxm.networks.VxmDense_128_256_256((128, 256, 256), nb_features, int_steps=0).to(device)
try:
    pretrained_state = torch.load(PRETRAINED_MODEL_PATH, map_location=device, weights_only=True)
except TypeError:
    pretrained_state = torch.load(PRETRAINED_MODEL_PATH, map_location=device)
model3D.load_state_dict(
    pretrained_state['model_state_dict'] if isinstance(pretrained_state, dict) and 'model_state_dict' in pretrained_state else pretrained_state
)
print(f'Loaded pretrained model: {PRETRAINED_MODEL_PATH.resolve()}')

optimizer = optim.Adam(model3D.parameters(), lr=LEARNING_RATE)
transformer = vxm.layers.SpatialTransformer((64, 128, 128)).to(device)
train_generator = different_patient_generator(volumes, lung_masks, train_ids, batch_size=BATCH_SIZE, seed=SPLIT_SEED)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

losses, image_losses, smoothness_losses, inverse_losses = [], [], [], []
validation_epochs, validation_mses, validation_maes, validation_inverses = [], [], [], []
best_validation_mse = float('inf')
best_checkpoint_path = CHECKPOINT_DIR / 'finetuned_different_patients_best_validation.pth'

for epoch in tqdm(range(1, FINETUNE_EPOCHS + 1), desc='Different-patient lung fine-tuning + inverse consistency'):
    moving_images, fixed_images, overlap_mask, _, _ = next(train_generator)
    moving_images = moving_images.to(device, dtype=torch.float32)
    fixed_images = fixed_images.to(device, dtype=torch.float32)
    overlap_mask = None if overlap_mask is None else overlap_mask.to(device, dtype=torch.float32)

    optimizer.zero_grad(set_to_none=True)
    transformed_image, flow_mf = reconstruct_warped(moving_images, fixed_images)
    transformed_fixed, flow_fm = reconstruct_warped(fixed_images, moving_images)
    image_loss = 0.5 * (
        masked_mse(fixed_images, transformed_image, overlap_mask)
        + masked_mse(moving_images, transformed_fixed, overlap_mask)
    )
    flow_smoothness = 0.5 * (smoothness_loss(flow_mf) + smoothness_loss(flow_fm))
    inverse_loss = inverse_consistency_loss(flow_mf, flow_fm)
    loss = (
        image_loss
        + SMOOTHNESS_WEIGHT * flow_smoothness
        + INVERSE_CONSISTENCY_WEIGHT * inverse_loss
    )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model3D.parameters(), max_norm=1.0)
    optimizer.step()

    losses.append(loss.detach().cpu().item())
    image_losses.append(image_loss.detach().cpu().item())
    smoothness_losses.append(flow_smoothness.detach().cpu().item())
    inverse_losses.append(inverse_loss.detach().cpu().item())

    if epoch % 100 == 0:
        print(
            f'Epoch {epoch:05d}/{FINETUNE_EPOCHS}: total={losses[-1]:.6f}, '
            f'image={image_losses[-1]:.6f}, smooth={smoothness_losses[-1]:.6f}, '
            f'inverse={inverse_losses[-1]:.6f}'
        )

    if epoch % VALIDATION_INTERVAL == 0:
        validation_mse, validation_mae, validation_inverse = evaluate_fixed_pairs(validation_pairs)
        validation_epochs.append(epoch)
        validation_mses.append(validation_mse)
        validation_maes.append(validation_mae)
        validation_inverses.append(validation_inverse)
        print(
            f'  validation: masked MSE={validation_mse:.6f}, '
            f'masked MAE={validation_mae:.6f}, inverse={validation_inverse:.6f}'
        )

        checkpoint = {
            'phase': 'different_patient_lung_inverse_consistent_finetuning',
            'epoch': epoch,
            'model_state_dict': model3D.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'validation_mse': validation_mse,
            'validation_mae': validation_mae,
            'validation_inverse_consistency': validation_inverse,
            'train_ids': train_ids.tolist(),
            'validation_ids': validation_ids.tolist(),
            'validation_pairs': validation_pairs,
            'archive_path': str(archive_path),
            'uses_lung_mask': lung_masks is not None,
            'config': {
                'learning_rate': LEARNING_RATE,
                'smoothness_weight': SMOOTHNESS_WEIGHT,
                'inverse_consistency_weight': INVERSE_CONSISTENCY_WEIGHT,
                'split_seed': SPLIT_SEED,
                'validation_fraction': VALIDATION_FRACTION,
            },
        }
        epoch_checkpoint_path = CHECKPOINT_DIR / f'finetune_epoch_{epoch:05d}.pth'
        torch.save(checkpoint, epoch_checkpoint_path)
        if validation_mse < best_validation_mse:
            best_validation_mse = validation_mse
            torch.save(checkpoint, best_checkpoint_path)
            print(f'  saved new best validation checkpoint: {best_checkpoint_path.name}')

    if epoch % VISUALIZATION_INTERVAL == 0:
        clear_output(wait=True)
        plt.figure(figsize=(10, 4))
        plt.plot(losses, label='total loss')
        plt.plot(image_losses, label='masked image loss')
        plt.plot(np.asarray(smoothness_losses) * SMOOTHNESS_WEIGHT, label='smoothness contribution')
        plt.plot(np.asarray(inverse_losses) * INVERSE_CONSISTENCY_WEIGHT, label='inverse consistency contribution')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Fine-tuning loss')
        plt.grid(alpha=0.3)
        plt.legend()
        plt.show()
        if validation_epochs:
            plt.figure(figsize=(8, 3.5))
            plt.plot(validation_epochs, validation_mses, marker='o', label='validation masked MSE')
            plt.plot(validation_epochs, validation_maes, marker='o', label='validation masked MAE')
            plt.plot(validation_epochs, validation_inverses, marker='o', label='validation inverse consistency')
            plt.xlabel('Epoch')
            plt.ylabel('Metric')
            plt.title('Held-out different-patient validation')
            plt.grid(alpha=0.3)
            plt.legend()
            plt.show()
        show_registration(moving_images, fixed_images, transformed_image, epoch)

final_checkpoint = {
    'phase': 'different_patient_lung_inverse_consistent_finetuning',
    'epoch': FINETUNE_EPOCHS,
    'model_state_dict': model3D.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_validation_mse': best_validation_mse,
    'last_validation_inverse_consistency': validation_inverses[-1] if validation_inverses else None,
    'train_ids': train_ids.tolist(),
    'validation_ids': validation_ids.tolist(),
    'validation_pairs': validation_pairs,
    'archive_path': str(archive_path),
    'uses_lung_mask': lung_masks is not None,
    'config': {
        'learning_rate': LEARNING_RATE,
        'smoothness_weight': SMOOTHNESS_WEIGHT,
        'inverse_consistency_weight': INVERSE_CONSISTENCY_WEIGHT,
        'split_seed': SPLIT_SEED,
        'validation_fraction': VALIDATION_FRACTION,
    },
}
final_checkpoint_path = CHECKPOINT_DIR / 'finetuned_different_patients_final.pth'
torch.save(final_checkpoint, final_checkpoint_path)
print(f'Last checkpoint: {final_checkpoint_path.resolve()}')
print(f'Best-validation checkpoint: {best_checkpoint_path.resolve()} (MSE={best_validation_mse:.6f})')


         Put a lung-masked archive in Data/ for true lung-focused fine-tuning.
Fine-tuning volume shape: (400, 128, 256, 256)
Train patients: 360, validation patients: 40
Fixed validation pairs: [(335, 30), (28, 282), (97, 360), (392, 242), (198, 52), (28, 335), (352, 152), (326, 173), (187, 207), (187, 198)]
[64, 128, 128]
Loaded pretrained model: C:\Users\ri0151fv\Saito\model_analysis_pipeline_pretrain.pth


ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

In [ ]:
import os

for drive in ["D:\\", "C:\\"]:
    for root, dirs, files in os.walk(drive):
        for file in files:
            if "wavelet" in file.lower() and file.endswith(".pth"):
                print(os.path.join(root, file))

D:\Saito\model_wavelet_128_256_256.pth
D:\Saito\model_wavelet_finetune_final.pth
D:\Saito\model_wavelet_pretrain_checkpoint.pth
D:\Saito\model_wavelet_pretrain_final.pth
D:\Saito\model_wavelet_pretrain_final_80000.pth
D:\Saito\model_wavelet_pretrain_restart_checkpoint.pth
D:\Saito\Saito_model_wavelet_128_256_256.pth
D:\Yamato\model_VXM_3D_MInoBed_WaveletEncorder.pth
D:\Yamato\model_VXM_3D_MInoBed_WaveletTest.pth
D:\Yamato\model_VXM_3D_weights_Wavelet.pth
C:\Users\user\OneDrive\ドキュメント\model_VXM_3D_MInoBed_WaveletTest.pth


In [ ]:
import torch, os

save_path = r"D:\Saito\model_wavelet_128_256_256.pth"
torch.save(model3D.state_dict(), save_path)

print(os.path.exists(save_path))
print(save_path)

True
D:\Saito\model_wavelet_128_256_256.pth


In [ ]:
# Check_Perfect_Reconstruction.py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt

pr_names = ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']

class CheckPRHaar3DAnalysis(nn.Module):
    def __init__(self):
        super().__init__()
        hL = torch.tensor([1.0, 1.0], dtype=torch.float32) / math.sqrt(2)
        hH = torch.tensor([1.0, -1.0], dtype=torch.float32) / math.sqrt(2)

        filters = []
        filter_names = []
        for z_name, z_filter in zip(['L', 'H'], [hL, hH]):
            for y_name, y_filter in zip(['L', 'H'], [hL, hH]):
                for x_name, x_filter in zip(['L', 'H'], [hL, hH]):
                    kernel = (
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )
                    filters.append(kernel)
                    filter_names.append(z_name + y_name + x_name)

        weight = torch.stack(filters, dim=0).unsqueeze(1)
        self.register_buffer('weight', weight)
        self.names = filter_names

    def forward(self, x):
        x_pad = F.pad(x, (0, 1, 0, 1, 0, 1))
        w = F.conv3d(x_pad, self.weight, stride=1, padding=0)
        return w

def check_pr_downsample_3d(w):
    return w[:, :, ::2, ::2, ::2]

def check_pr_upsample_3d(w_down):
    B, C, D, H, W = w_down.shape
    w_up = torch.zeros(B, C, D * 2, H * 2, W * 2, dtype=w_down.dtype, device=w_down.device)
    w_up[:, :, ::2, ::2, ::2] = w_down
    return w_up

def check_pr_make_3d_filter(fz, fy, fx):
    return fz[:, None, None] * fy[None, :, None] * fx[None, None, :]

class CheckPRHaar3DSynthesis(nn.Module):
    def __init__(self):
        super().__init__()
        sqrt2 = math.sqrt(2.0)
        low = torch.tensor([1.0, 1.0], dtype=torch.float32) / sqrt2
        high = torch.tensor([1.0, -1.0], dtype=torch.float32) / sqrt2

        filters = torch.stack([
            check_pr_make_3d_filter(low, low, low),
            check_pr_make_3d_filter(low, low, high),
            check_pr_make_3d_filter(low, high, low),
            check_pr_make_3d_filter(low, high, high),
            check_pr_make_3d_filter(high, low, low),
            check_pr_make_3d_filter(high, low, high),
            check_pr_make_3d_filter(high, high, low),
            check_pr_make_3d_filter(high, high, high),
        ], dim=0)

        filters = torch.flip(filters, dims=[1, 2, 3]).unsqueeze(1)
        self.register_buffer('filters', filters)

    def forward(self, w_up):
        B, C, D, H, W = w_up.shape
        filtered_bands = []
        for i, name in enumerate(pr_names):
            band = w_up[:, i:i + 1, :, :, :]
            kernel = self.filters[i:i + 1]
            filtered = F.conv3d(band, kernel, stride=1, padding=1)
            filtered = filtered[:, :, :D, :H, :W]
            filtered_bands.append(filtered)
            print(name, 'Synthesis後:', filtered.shape)

        filtered_bands = torch.cat(filtered_bands, dim=1)
        reconstructed = torch.sum(filtered_bands, dim=1, keepdim=True)
        return filtered_bands, reconstructed

def check_pr_frequency_response(h, omega):
    n = np.arange(len(h))
    response = np.sum(h[None, :] * np.exp(-1j * omega[:, None] * n[None, :]), axis=1)
    return response

In [ ]:
# Check_Perfect_Reconstruction.py: reconstruction check
check_pr_x = torch.as_tensor(
    x_train[0:1],
    dtype=torch.float32,
    device=device
).unsqueeze(1)

print('\n===================================')
print('入力')
print('===================================')
print('元画像:', check_pr_x.shape)

print('\n===================================')
print('1. Analysis Filter')
print('===================================')
check_pr_analysis = CheckPRHaar3DAnalysis().to(device)
check_pr_w = check_pr_analysis(check_pr_x)
print('Analysis後:', check_pr_w.shape)
print('周波数成分:', check_pr_analysis.names)

print('\n===================================')
print('2. Downsampling')
print('===================================')
check_pr_w_down = check_pr_downsample_3d(check_pr_w)
print('Downsampling後:', check_pr_w_down.shape)

print('\n===================================')
print('3. Upsampling')
print('===================================')
check_pr_w_up = check_pr_upsample_3d(check_pr_w_down)
print('Upsampling後:', check_pr_w_up.shape)

check_pr_up_error = torch.abs(check_pr_w_up[:, :, ::2, ::2, ::2] - check_pr_w_down)
print('Upsampling配置確認 平均誤差:', check_pr_up_error.mean().item())
print('Upsampling配置確認 最大誤差:', check_pr_up_error.max().item())

print('\n===================================')
print('4. Synthesis Filter')
print('===================================')
check_pr_synthesis = CheckPRHaar3DSynthesis().to(device)
check_pr_filtered_bands, check_pr_reconstructed = check_pr_synthesis(check_pr_w_up)
print('\nSynthesis後8成分:', check_pr_filtered_bands.shape)
print('再構成画像:', check_pr_reconstructed.shape)

print('\n===================================')
print('5. Reconstruction Error')
print('===================================')

if check_pr_x.shape != check_pr_reconstructed.shape:
    print('サイズが一致していません')
    print('Original:', check_pr_x.shape)
    print('Reconstructed:', check_pr_reconstructed.shape)
else:
    print('サイズ一致')
    check_pr_diff = check_pr_x - check_pr_reconstructed
    check_pr_absolute_error = torch.abs(check_pr_diff)
    check_pr_mae = torch.mean(check_pr_absolute_error)
    check_pr_max_error = torch.max(check_pr_absolute_error)
    check_pr_mse = torch.mean(check_pr_diff ** 2)
    check_pr_relative_error = torch.norm(check_pr_diff) / torch.norm(check_pr_x)
    print('\n===== 再構成誤差 =====')
    print('MAE:', check_pr_mae.item())
    print('最大絶対誤差:', check_pr_max_error.item())
    print('MSE:', check_pr_mse.item())
    print('相対誤差:', check_pr_relative_error.item())

    if check_pr_max_error.item() > 0:
         check_pr_error_order = int(np.floor(np.log10(check_pr_max_error.item())))
         check_pr_error_scale = 10.0 ** (-check_pr_error_order)
    else:
         check_pr_error_order = 0
         check_pr_error_scale = 1.0

    print('誤差表示スケール:', f'x 1e{-check_pr_error_order}' if check_pr_max_error.item() > 0 else 'no scaling')

print('\n===================================')
print('6. Save Results')
print('===================================')
np.save(r'D:\Saito\wavelet_reconstructed.npy', check_pr_reconstructed.detach().cpu().numpy())
print('再構成画像を保存しました')

if check_pr_x.shape == check_pr_reconstructed.shape:
    check_pr_slice_index = check_pr_x.shape[2] // 2

    plt.figure(figsize=(8, 8))
    plt.imshow(check_pr_x[0, 0, check_pr_slice_index].detach().cpu().numpy(), cmap='gray')
    plt.title('Original Image')
    plt.axis('off')
    plt.savefig(r'D:\Saito\wavelet_original.png', dpi=300, bbox_inches='tight')
    plt.close()

    plt.figure(figsize=(8, 8))
    plt.imshow(check_pr_reconstructed[0, 0, check_pr_slice_index].detach().cpu().numpy(), cmap='gray')
    plt.title('Reconstructed Image')
    plt.axis('off')
    plt.savefig(r'D:\Saito\wavelet_reconstructed.png', dpi=300, bbox_inches='tight')
    plt.close()

    plt.figure(figsize=(8, 8))
    plt.imshow(
         check_pr_absolute_error[0, 0, check_pr_slice_index].detach().cpu().numpy(),
         cmap='inferno',
         vmin=0.0,
         vmax=check_pr_max_error.item()
     )
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title('Absolute Reconstruction Error')
    plt.axis('off')
    plt.savefig(r'D:\Saito\wavelet_error.png', dpi=300, bbox_inches='tight')
    plt.close()

    plt.figure(figsize=(8, 8))
    plt.imshow(
       (check_pr_absolute_error[0, 0, check_pr_slice_index] * check_pr_error_scale).detach().cpu().numpy(),
       cmap='inferno'
     )
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title(
         f'Scaled Error (x 1e{-check_pr_error_order})'
         if check_pr_max_error.item() > 0
         else 'Scaled Error'
     )
    plt.axis('off')
    plt.savefig(r'D:\Saito\wavelet_error_scaled.png', dpi=300, bbox_inches='tight')
    plt.close()


入力
元画像: torch.Size([1, 1, 128, 256, 256])

1. Analysis Filter
Analysis後: torch.Size([1, 8, 128, 256, 256])
周波数成分: ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']

2. Downsampling
Downsampling後: torch.Size([1, 8, 64, 128, 128])

3. Upsampling
Upsampling後: torch.Size([1, 8, 128, 256, 256])
Upsampling配置確認 平均誤差: 0.0
Upsampling配置確認 最大誤差: 0.0

4. Synthesis Filter
LLL Synthesis後: torch.Size([1, 1, 128, 256, 256])
LLH Synthesis後: torch.Size([1, 1, 128, 256, 256])
LHL Synthesis後: torch.Size([1, 1, 128, 256, 256])
LHH Synthesis後: torch.Size([1, 1, 128, 256, 256])
HLL Synthesis後: torch.Size([1, 1, 128, 256, 256])
HLH Synthesis後: torch.Size([1, 1, 128, 256, 256])
HHL Synthesis後: torch.Size([1, 1, 128, 256, 256])
HHH Synthesis後: torch.Size([1, 1, 128, 256, 256])

Synthesis後8成分: torch.Size([1, 8, 128, 256, 256])
再構成画像: torch.Size([1, 1, 128, 256, 256])

5. Reconstruction Error
サイズ一致

===== 再構成誤差 =====
MAE: 3.967887707290174e-08
最大絶対誤差: 4.172325134277344e-07
MSE: 5.188463472473011e-15
相対誤差:

In [ ]:
if check_pr_x.shape == check_pr_reconstructed.shape:
    check_pr_slice_index = check_pr_x.shape[2] // 2

    # ========================================
    # 元画像
    # ========================================
    plt.figure(figsize=(8, 8))

    plt.imshow(
        check_pr_x[
            0,
            0,
            check_pr_slice_index
        ].detach().cpu().numpy(),
        cmap='gray'
    )

    plt.title('Original Image')
    plt.axis('off')

    plt.savefig(
        r'D:\Saito\wavelet_original.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()


    # ========================================
    # 再構成画像
    # ========================================
    plt.figure(figsize=(8, 8))

    plt.imshow(
        check_pr_reconstructed[
            0,
            0,
            check_pr_slice_index
        ].detach().cpu().numpy(),
        cmap='gray'
    )

    plt.title('Reconstructed Image')
    plt.axis('off')

    plt.savefig(
        r'D:\Saito\wavelet_reconstructed.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()


    # ========================================
    # 絶対誤差画像
    # ========================================
    plt.figure(figsize=(8, 8))

    plt.imshow(
        check_pr_absolute_error[
            0,
            0,
            check_pr_slice_index
        ].detach().cpu().numpy(),
        cmap='inferno',
        vmin=0.0,
        vmax=check_pr_max_error.item()
    )

    plt.colorbar(
        fraction=0.046,
        pad=0.04
    )

    plt.title('Absolute Reconstruction Error')
    plt.axis('off')

    plt.savefig(
        r'D:\Saito\wavelet_error.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()


    # ========================================
    # 拡大表示した誤差画像
    # ========================================
    plt.figure(figsize=(8, 8))

    check_pr_scaled_error = (
        check_pr_absolute_error[
            0,
            0,
            check_pr_slice_index
        ]
        * check_pr_error_scale
    ).detach().cpu().numpy()

    plt.imshow(
        check_pr_scaled_error,
        cmap='inferno'
    )

    plt.colorbar(
        fraction=0.046,
        pad=0.04
    )

    if check_pr_max_error.item() > 0:
        plt.title(
            f'Scaled Error (x 1e{-check_pr_error_order})'
        )
    else:
        plt.title('Scaled Error')

    plt.axis('off')

    plt.savefig(
        r'D:\Saito\wavelet_error_scaled.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

    print('画像を保存しました')

画像を保存しました


In [ ]:
# Check_Perfect_Reconstruction.py: perfect reconstruction conditions
print('\n===================================')
print('7. Perfect Reconstruction Conditions')
print('===================================')

check_pr_sqrt2 = np.sqrt(2.0)
check_pr_h0 = np.array([1.0, 1.0]) / check_pr_sqrt2
check_pr_h1 = np.array([-1.0, 1.0]) / check_pr_sqrt2
check_pr_f0 = np.array([1.0, 1.0]) / check_pr_sqrt2
check_pr_f1 = np.array([1.0, -1.0]) / check_pr_sqrt2

check_pr_N = 2048
check_pr_omega = np.linspace(-np.pi, np.pi, check_pr_N, endpoint=False)

check_pr_H0 = check_pr_frequency_response(check_pr_h0, check_pr_omega)
check_pr_H1 = check_pr_frequency_response(check_pr_h1, check_pr_omega)
check_pr_F0 = check_pr_frequency_response(check_pr_f0, check_pr_omega)
check_pr_F1 = check_pr_frequency_response(check_pr_f1, check_pr_omega)
check_pr_H0_minus = check_pr_frequency_response(check_pr_h0, check_pr_omega + np.pi)
check_pr_H1_minus = check_pr_frequency_response(check_pr_h1, check_pr_omega + np.pi)

check_pr_alias_term = check_pr_H0_minus * check_pr_F0 + check_pr_H1_minus * check_pr_F1
check_pr_alias_max = np.max(np.abs(check_pr_alias_term))
print('\n-----------------------------------')
print('条件1: Alias Cancellation')
print('-----------------------------------')
print('最大Alias成分:', check_pr_alias_max)

check_pr_T = check_pr_H0 * check_pr_F0 + check_pr_H1 * check_pr_F1
check_pr_magnitude = np.abs(check_pr_T)
check_pr_magnitude_min = np.min(check_pr_magnitude)
check_pr_magnitude_max = np.max(check_pr_magnitude)
check_pr_magnitude_variation = check_pr_magnitude_max - check_pr_magnitude_min
print('\n-----------------------------------')
print('条件2: Amplitude Distortion')
print('-----------------------------------')
print('Magnitude min:', check_pr_magnitude_min)
print('Magnitude max:', check_pr_magnitude_max)
print('Magnitude variation:', check_pr_magnitude_variation)

check_pr_phase = np.unwrap(np.angle(check_pr_T))
check_pr_phase_coef = np.polyfit(check_pr_omega, check_pr_phase, 1)
check_pr_phase_fit = np.polyval(check_pr_phase_coef, check_pr_omega)
check_pr_phase_error = check_pr_phase - check_pr_phase_fit
check_pr_max_phase_error = np.max(np.abs(check_pr_phase_error))
print('\n-----------------------------------')
print('条件3: Phase Distortion')
print('-----------------------------------')
print('Phase slope:', check_pr_phase_coef[0])
print('最大直線位相誤差:', check_pr_max_phase_error)

check_pr_tolerance = 1e-10
print('\n===================================')
print('PR Condition Results')
print('===================================')
print('条件1 Alias Cancellation:', 'OK' if check_pr_alias_max < check_pr_tolerance else 'NG')
print('条件2 Amplitude Distortion:', 'OK' if check_pr_magnitude_variation < check_pr_tolerance else 'NG')
print('条件3 Phase Distortion:', 'OK' if check_pr_max_phase_error < check_pr_tolerance else 'NG')

plt.figure(figsize=(8, 5))
plt.plot(check_pr_omega, np.abs(check_pr_alias_term))
plt.xlabel('Angular Frequency ω')
plt.ylabel('|Alias Term|')
plt.title('Alias Cancellation')
plt.grid()
plt.savefig(r'D:\Saito\pr_alias.png', dpi=300, bbox_inches='tight')
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(check_pr_omega, check_pr_magnitude)
plt.xlabel('Angular Frequency ω')
plt.ylabel('|T(e^jω)|')
plt.title('Distortion Transfer Function Magnitude')
plt.grid()
plt.savefig(r'D:\Saito\pr_amplitude.png', dpi=300, bbox_inches='tight')
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(check_pr_omega, check_pr_phase, label='Actual Phase')
plt.plot(check_pr_omega, check_pr_phase_fit, linestyle='--', label='Linear Fit')
plt.xlabel('Angular Frequency ω')
plt.ylabel('Phase [rad]')
plt.title('Phase Response')
plt.legend()
plt.grid()
plt.savefig(r'D:\Saito\pr_phase.png', dpi=300, bbox_inches='tight')
plt.close()

print('\nPR条件確認用グラフを保存しました')
print('\n処理完了')


7. Perfect Reconstruction Conditions

-----------------------------------
条件1: Alias Cancellation
-----------------------------------
最大Alias成分: 3.554447978966673e-16

-----------------------------------
条件2: Amplitude Distortion
-----------------------------------
Magnitude min: 1.999999999999999
Magnitude max: 2.0000000000000004
Magnitude variation: 1.5543122344752192e-15

-----------------------------------
条件3: Phase Distortion
-----------------------------------
Phase slope: -1.0
最大直線位相誤差: 8.881784197001252e-16

PR Condition Results
条件1 Alias Cancellation: OK
条件2 Amplitude Distortion: OK
条件3 Phase Distortion: OK

PR条件確認用グラフを保存しました

処理完了


In [ ]:
# ファインチューニングは上の3万epochセルに統合済みです。ここでは何もしません。

In [ ]:
# チェックポイント保存は上の3万epochセルで100epochごとに実行されます。

In [ ]:
# 最終モデル保存は上の3万epochセルの完了時に実行されます。